In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from sinkpro import SinkhornProcrustes
import pickle

In [ ]:
with open('blade_embedding_data.pkl', 'rb') as f:
    data = pickle.load(f)

In [ ]:
frames = []
for key in data:
    frame_data = data[key]
    embeddings_dict = frame_data[0]
    embeddings_dict = {int(i): emb for i, emb in embeddings_dict.items()}
    graph = frame_data[2]
    nodes = np.array(list(graph.nodes())).astype(int)
    edges = np.array(list(graph.edges())).astype(int)
    embeddings = np.zeros((len(nodes), 2), dtype=float)
    node_to_index = {node: idx for idx, node in enumerate(nodes)}
    for node, emb in embeddings_dict.items():
        idx = node_to_index[node]
        embeddings[idx] = emb
    frames.append((nodes, edges, embeddings))

In [ ]:
for frame_idx in range(len(frames) - 1):
    nodes1, edges1, emb1 = frames[frame_idx]
    nodes2, edges2, emb2 = frames[frame_idx + 10]
    break

In [ ]:
spro = SinkhornProcrustes(emb2, emb1, fit_scale=True, coupling="sinkhorn")

In [ ]:
spro_res = spro.run(1.0, 1e-3, 20, 50, eps_log_decay = True, return_score = True, return_weight_matrix = True)

In [ ]:
embd2_t = spro.transform(emb2, spro_res[0], spro_res[1], spro_res[2])

In [ ]:
spro_res[-1].sum(axis=0)

In [ ]:
plt.scatter(emb1[:, 0], emb1[:, 1], color='blue', label='Frame 1')
plt.scatter(embd2_t[:, 0], embd2_t[:, 1], color='red', label='Frame 2', alpha=0.5)
plt.legend()
plt.show()